# Importing Libraries and Datasets

In [1]:
# Import necessary library
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from datetime import datetime, timedelta

In [2]:
# Excel file path
excel_file_path = '../data/LifecycleDS-Exercise-sampledata.xlsx'

# Load each sheet into a separate DataFrame
df_onboarding = pd.read_excel(excel_file_path, sheet_name='team_onboarding_experiment')
df_prod_activity = pd.read_excel(excel_file_path, sheet_name='team_prod_activity_log')
df_financial = pd.read_excel(excel_file_path, sheet_name='team_financial_metrics')

# Shape check to count rows and columns
print("Onboarding data shape:", df_onboarding.shape)
print("Prod activity data shape:", df_prod_activity.shape)
print("Financial data shape:", df_financial.shape)

Onboarding data shape: (120, 5)
Prod activity data shape: (250, 5)
Financial data shape: (20, 4)


### team_onboarding_experiment dataset

In [3]:
df_onboarding.head()

,team_id,creator_user_id,creation_timestamp_utc,email_journey,date_first_plus_one_joiner_utc
0,T001,U1001,2023-09-01 10:00:00,Control,2023-09-04
1,T002,U1002,2023-09-01 10:05:00,Control,2023-09-08
2,T003,U1003,2023-09-01 10:10:00,Control,NaT
3,T004,U1004,2023-09-01 10:15:00,Control,2023-09-06
4,T005,U1005,2023-09-01 10:20:00,Control,NaT


### team_prod_activity_log dataset

In [4]:
df_prod_activity

,activity_id,team_id,user_id,activity_timestamp_utc,feature_used
0,A001,T001,U1001,2023-09-01 10:02:00,Sent_Message
1,A002,T041,U1041,2023-09-04 13:25:00,Created_Channel
2,A003,T081,U1081,2023-09-07 16:45:00,Sent_Message
3,A004,T045,U1045,2023-09-05 10:30:00,Created_Canvas
4,A005,T002,U1002,2023-09-02 11:30:00,Used_Search
...,...,...,...,...,...
245,A246,T012,U1012,2023-10-01 11:00:00,Created_Channel
246,A247,T013,U1013,2023-10-01 12:00:00,Sent_Message
247,A248,T015,U1015,2023-10-01 13:00:00,Created_Channel
248,A249,T017,U1017,2023-10-01 14:00:00,Sent_Message


# Primary Metric

# Calculating % of Teams with +1 joiner and Statistical Tests

In [18]:
teams_with_joiner = df_onboarding.groupby('email_journey').agg({
    'team_id': 'count',  # Total teams
    'date_first_plus_one_joiner_utc': lambda x: x.notna().sum()  # Teams with +1
}).reset_index()

teams_with_joiner.columns = ['email_journey', 'total_teams', 'teams_with_joiner']
teams_with_joiner['pct_with_joiner'] = (teams_with_joiner['teams_with_joiner'] / 
                                         teams_with_joiner['total_teams'] * 100).round(1)

print("\n% of Teams with +1 Joiner:")
print(teams_with_joiner)


% of Teams with +1 Joiner:
  email_journey  total_teams  teams_with_joiner  pct_with_joiner
0       Control           40                 26             65.0
1     Variant_A           40                 31             77.5
2     Variant_B           40                 27             67.5


# STATISTICAL TESTS

In [19]:
# Create contingency table
contingency_table = pd.DataFrame({
    'With_Joiner': teams_with_joiner['teams_with_joiner'].values,
    'Without_Joiner': (teams_with_joiner['total_teams'] - teams_with_joiner['teams_with_joiner']).values
}, index=teams_with_joiner['email_journey'])

print("\nContingency Table:")
print(contingency_table)

# Chi-squared test (overall - tests all 3 groups)
chi2, p_value_chi, dof, expected = stats.chi2_contingency(contingency_table.T)

print(f"\nChi-squared test (3 groups):")
print(f"  Chi² = {chi2:.3f}")
print(f"  p-value = {p_value_chi:.4f}")
print(f"  Significant: {'Yes' if p_value_chi < 0.05 else 'No'}")

# Pairwise proportion z-tests
from statsmodels.stats.proportion import proportions_ztest

# Extract values for each group
control_idx = teams_with_joiner[teams_with_joiner['email_journey'] == 'Control'].index[0]
variant_a_idx = teams_with_joiner[teams_with_joiner['email_journey'] == 'Variant_A'].index[0]
variant_b_idx = teams_with_joiner[teams_with_joiner['email_journey'] == 'Variant_B'].index[0]

control_successes = teams_with_joiner.loc[control_idx, 'teams_with_joiner']
control_n = teams_with_joiner.loc[control_idx, 'total_teams']
control_pct = teams_with_joiner.loc[control_idx, 'pct_with_joiner']

variant_a_successes = teams_with_joiner.loc[variant_a_idx, 'teams_with_joiner']
variant_a_n = teams_with_joiner.loc[variant_a_idx, 'total_teams']
variant_a_pct = teams_with_joiner.loc[variant_a_idx, 'pct_with_joiner']

variant_b_successes = teams_with_joiner.loc[variant_b_idx, 'teams_with_joiner']
variant_b_n = teams_with_joiner.loc[variant_b_idx, 'total_teams']
variant_b_pct = teams_with_joiner.loc[variant_b_idx, 'pct_with_joiner']

# Variant A vs Control
z_stat_a, p_value_a = proportions_ztest(
    count=[variant_a_successes, control_successes],
    nobs=[variant_a_n, control_n],
    alternative='two-sided'
)

print(f"\n--- Pairwise Comparisons ---")
print(f"\nVariant A vs Control (proportion z-test):")
print(f"  Control: {control_successes}/{control_n} = {control_pct:.1f}%")
print(f"  Variant A: {variant_a_successes}/{variant_a_n} = {variant_a_pct:.1f}%")
print(f"  Absolute difference: {variant_a_pct - control_pct:+.1f} percentage points")
print(f"  Relative lift: {((variant_a_pct - control_pct) / control_pct * 100):+.1f}%")
print(f"  Z-statistic = {z_stat_a:.3f}")
print(f"  p-value = {p_value_a:.4f}")
print(f"  Significant: {'Yes' if p_value_a < 0.05 else 'No'}")

# Variant B vs Control
z_stat_b, p_value_b = proportions_ztest(
    count=[variant_b_successes, control_successes],
    nobs=[variant_b_n, control_n],
    alternative='two-sided'
)

print(f"\nVariant B vs Control (proportion z-test):")
print(f"  Control: {control_successes}/{control_n} = {control_pct:.1f}%")
print(f"  Variant B: {variant_b_successes}/{variant_b_n} = {variant_b_pct:.1f}%")
print(f"  Absolute difference: {variant_b_pct - control_pct:+.1f} percentage points")
print(f"  Relative lift: {((variant_b_pct - control_pct) / control_pct * 100):+.1f}%")
print(f"  Z-statistic = {z_stat_b:.3f}")
print(f"  p-value = {p_value_b:.4f}")
print(f"  Significant: {'Yes' if p_value_b < 0.05 else 'No'}")

# Effect size (Cohen's h for proportions)
def cohens_h(p1, p2):
    """Calculate Cohen's h for two proportions"""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

h_a = cohens_h(variant_a_pct/100, control_pct/100)
h_b = cohens_h(variant_b_pct/100, control_pct/100)

print(f"\n--- Effect Sizes (Cohen's h) ---")
print(f"Variant A vs Control: h = {h_a:.3f} ({'small' if abs(h_a) < 0.5 else 'medium' if abs(h_a) < 0.8 else 'large'})")
print(f"Variant B vs Control: h = {h_b:.3f} ({'small' if abs(h_b) < 0.5 else 'medium' if abs(h_b) < 0.8 else 'large'})")


Contingency Table:
               With_Joiner  Without_Joiner
email_journey                             
Control                 26              14
Variant_A               31               9
Variant_B               27              13

Chi-squared test (3 groups):
  Chi² = 1.667
  p-value = 0.4346
  Significant: No

--- Pairwise Comparisons ---

Variant A vs Control (proportion z-test):
  Control: 26/40 = 65.0%
  Variant A: 31/40 = 77.5%
  Absolute difference: +12.5 percentage points
  Relative lift: +19.2%
  Z-statistic = 1.235
  p-value = 0.2168
  Significant: No

Variant B vs Control (proportion z-test):
  Control: 26/40 = 65.0%
  Variant B: 27/40 = 67.5%
  Absolute difference: +2.5 percentage points
  Relative lift: +3.8%
  Z-statistic = 0.236
  p-value = 0.8131
  Significant: No

--- Effect Sizes (Cohen's h) ---
Variant A vs Control: h = 0.278 (small)
Variant B vs Control: h = 0.053 (small)


### Merging on team_id to get Average Features Used metric

In [20]:
# Merge activity with onboarding to know email_journey per team
df_activity_merged = df_prod_activity.merge(df_onboarding[['team_id','email_journey']], on='team_id', how='left')
df_activity_merged.sort_values(by='team_id')


,activity_id,team_id,user_id,activity_timestamp_utc,feature_used,email_journey
0,A001,T001,U1001,2023-09-01 10:02:00,Sent_Message,Control
139,A140,T001,U1001,2023-09-23 17:00:00,Created_Canvas,Control
189,A190,T001,U1001,2023-09-26 19:00:00,Created_Canvas,Control
239,A240,T001,U1001,2023-09-29 21:00:00,Used_Search,Control
65,A066,T001,U1001,2023-09-02 11:00:00,Used_Search,Control
...,...,...,...,...,...,...
94,A095,T113,U1113,2023-09-20 18:00:00,Used_Search,Variant_B
184,A185,T115,U1115,2023-09-26 14:00:00,Used_Search,Variant_B
234,A235,T115,U1115,2023-09-29 16:00:00,Created_Canvas,Variant_B
134,A135,T115,U1115,2023-09-23 12:00:00,Used_Search,Variant_B


### Grouping by team and journey allows us to count unique features used (breadth) and total features used (depth)

In [21]:
# Count unique features used per team in first 30 days

unique_feature_usage = df_activity_merged.groupby(['team_id','email_journey'])['feature_used'].nunique().reset_index()
unique_feature_usage

,team_id,email_journey,feature_used
0,T001,Control,3
1,T002,Control,1
2,T004,Control,3
3,T006,Control,3
4,T007,Control,3
...,...,...,...
58,T106,Variant_B,2
59,T107,Variant_B,3
60,T109,Variant_B,3
61,T113,Variant_B,2


In [22]:
teams_using_features = unique_feature_usage.groupby('email_journey')['team_id'].nunique()
teams_using_features

email_journey
Control      17
Variant_A    28
Variant_B    18
Name: team_id, dtype: int64

In [23]:
total_teams = df_onboarding.groupby('email_journey')['team_id'].count()
total_teams

email_journey
Control      40
Variant_A    40
Variant_B    40
Name: team_id, dtype: int64

In [24]:
# Calculate adoption rates
adoption_rates = (teams_using_features / total_teams).reset_index()
adoption_rates.columns = ['email_journey', 'adoption_rate']
print("\nFeature Adoption Rates:")
print(adoption_rates)


Feature Adoption Rates:
  email_journey  adoption_rate
0       Control          0.425
1     Variant_A          0.700
2     Variant_B          0.450


# Feature Adoption Statistical Tests

In [25]:
# Chi-squared test for independence (all 3 groups)
contingency_table = pd.DataFrame({
    'Used_Features': teams_using_features,
    'No_Features': total_teams - teams_using_features
})
print("\nContingency Table:")
print(contingency_table)

chi2, p_value_chi, dof, expected = stats.chi2_contingency(contingency_table.T)
print(f"\nChi-squared test (3 groups):")
print(f"  Chi2 = {chi2:.3f}")
print(f"  p-value = {p_value_chi:.4f}")
print(f"  Significant: {'Yes' if p_value_chi < 0.05 else 'No'}")


Contingency Table:
               Used_Features  No_Features
email_journey                            
Control                   17           23
Variant_A                 28           12
Variant_B                 18           22

Chi-squared test (3 groups):
  Chi2 = 7.419
  p-value = 0.0245
  Significant: Yes


In [26]:
# Pairwise Z-tests for proportions (Variant A vs Control, Variant B vs Control)
from statsmodels.stats.proportion import proportions_ztest

# Variant A vs Control
control_successes = teams_using_features.loc['Control']
control_n = total_teams.loc['Control']
variant_a_successes = teams_using_features.loc['Variant_A']
variant_a_n = total_teams.loc['Variant_A']

z_stat_a, p_value_a = proportions_ztest(
    count=[variant_a_successes, control_successes],
    nobs=[variant_a_n, control_n],
    alternative='two-sided'
)

print(f"\nVariant A vs Control (proportion z-test):")
print(f"  Control: {control_successes}/{control_n} = {control_successes/control_n:.1%}")
print(f"  Variant A: {variant_a_successes}/{variant_a_n} = {variant_a_successes/variant_a_n:.1%}")
print(f"  Z-statistic = {z_stat_a:.3f}")
print(f"  p-value = {p_value_a:.4f}")

# Variant B vs Control
variant_b_successes = teams_using_features.loc['Variant_B']
variant_b_n = total_teams.loc['Variant_B']

z_stat_b, p_value_b = proportions_ztest(
    count=[variant_b_successes, control_successes],
    nobs=[variant_b_n, control_n],
    alternative='two-sided'
)

print(f"\nVariant B vs Control (proportion z-test):")
print(f"  Control: {control_successes}/{control_n} = {control_successes/control_n:.1%}")
print(f"  Variant B: {variant_b_successes}/{variant_b_n} = {variant_b_successes/variant_b_n:.1%}")
print(f"  Z-statistic = {z_stat_b:.3f}")
print(f"  p-value = {p_value_b:.4f}")


Variant A vs Control (proportion z-test):
  Control: 17/40 = 42.5%
  Variant A: 28/40 = 70.0%
  Z-statistic = 2.479
  p-value = 0.0132

Variant B vs Control (proportion z-test):
  Control: 17/40 = 42.5%
  Variant B: 18/40 = 45.0%
  Z-statistic = 0.225
  p-value = 0.8217


In [13]:
# Count unique features used per team in first 30 days

feature_adoption = unique_feature_usage.groupby('email_journey')['feature_used'].mean().reset_index()
feature_adoption.rename(columns={'feature_used':'avg_features_used'}, inplace=True)
print(feature_adoption)

  email_journey  avg_features_used
0       Control           2.764706
1     Variant_A           2.607143
2     Variant_B           2.166667


In [27]:
# Get the team-level feature counts
features_per_team = unique_feature_usage.groupby(['team_id', 'email_journey'])['feature_used'].sum().reset_index()
features_per_team.rename(columns={'feature_used': 'num_features'}, inplace=True)

# Split by journey
control_features = features_per_team[features_per_team['email_journey'] == 'Control']['num_features']
variant_a_features = features_per_team[features_per_team['email_journey'] == 'Variant_A']['num_features']
variant_b_features = features_per_team[features_per_team['email_journey'] == 'Variant_B']['num_features']

print(f"Control mean: {control_features.mean():.6f}")  
print(f"Variant A mean: {variant_a_features.mean():.6f}")
print(f"Variant B mean: {variant_b_features.mean():.6f}")

# One-way ANOVA
f_stat, p_value_anova = stats.f_oneway(control_features, variant_a_features, variant_b_features)
print(f"\nOne-way ANOVA:")
print(f"  F = {f_stat:.3f}, p = {p_value_anova:.4f}")

# Pairwise t-tests
t_stat_a, p_value_a = stats.ttest_ind(variant_a_features, control_features)
print(f"\nVariant A vs Control: t = {t_stat_a:.3f}, p = {p_value_a:.4f}")

t_stat_b, p_value_b = stats.ttest_ind(variant_b_features, control_features)
print(f"Variant B vs Control: t = {t_stat_b:.3f}, p = {p_value_b:.4f}")

Control mean: 2.764706
Variant A mean: 2.607143
Variant B mean: 2.166667

One-way ANOVA:
  F = 4.304, p = 0.0179

Variant A vs Control: t = -0.798, p = 0.4293
Variant B vs Control: t = -2.759, p = 0.0094


In [28]:
# Count total features used per variant in first 30 days
# Since each variant could have a different amount of teams using adopting features, this metric is not recommended
total_feature_usage = df_activity_merged.groupby(['team_id','email_journey'])['feature_used'].count().reset_index()

feature_adoption = total_feature_usage.groupby('email_journey')['feature_used'].mean().reset_index()
feature_adoption.rename(columns={'feature_used':'avg_features_used'}, inplace=True)
print(feature_adoption)

  email_journey  avg_features_used
0       Control           4.294118
1     Variant_A           4.607143
2     Variant_B           2.666667


# Merge Financial Data

# Paid Conversion Rate (PCR)

In [29]:
# Merge financial data with onboarding
df_onboarding['converted'] = df_onboarding['team_id'].isin(df_financial['team_id'])

pcr = df_onboarding.groupby('email_journey')['converted'].mean().reset_index()
pcr.rename(columns={'converted':'paid_conversion_rate'}, inplace=True)
print(pcr)

  email_journey  paid_conversion_rate
0       Control                  0.10
1     Variant_A                  0.25
2     Variant_B                  0.15


 # Paid Conversion Rate (PCR) Statistical Test

In [30]:
# Count conversions by journey
conversion_counts = df_onboarding.groupby('email_journey').agg({
    'team_id': 'count',
    'converted': 'sum'
}).reset_index()
conversion_counts.columns = ['email_journey', 'total_teams', 'converted_teams']

print("\nConversion Counts:")
print(conversion_counts)

# Create contingency table
contingency_table = pd.DataFrame({
    'Converted': conversion_counts['converted_teams'].values,
    'Not_Converted': (conversion_counts['total_teams'] - conversion_counts['converted_teams']).values
}, index=conversion_counts['email_journey'])

print("\nContingency Table:")
print(contingency_table)


Conversion Counts:
  email_journey  total_teams  converted_teams
0       Control           40                4
1     Variant_A           40               10
2     Variant_B           40                6

Contingency Table:
               Converted  Not_Converted
email_journey                          
Control                4             36
Variant_A             10             30
Variant_B              6             34


# CHI-SQUARED TEST (Overall test for all 3 groups)

In [31]:
chi2, p_value_chi, dof, expected = stats.chi2_contingency(contingency_table.T)

print(f"\nChi-squared test (3 groups):")
print(f"  Chi² = {chi2:.3f}")
print(f"  p-value = {p_value_chi:.4f}")
print(f"  Significant: {'Yes' if p_value_chi < 0.05 else 'No'}")




Chi-squared test (3 groups):
  Chi² = 3.360
  p-value = 0.1864
  Significant: No


# PAIRWISE PROPORTION Z-TESTS

In [32]:
from statsmodels.stats.proportion import proportions_ztest

# Extract values
control_idx = conversion_counts[conversion_counts['email_journey'] == 'Control'].index[0]
variant_a_idx = conversion_counts[conversion_counts['email_journey'] == 'Variant_A'].index[0]
variant_b_idx = conversion_counts[conversion_counts['email_journey'] == 'Variant_B'].index[0]

control_conv = conversion_counts.loc[control_idx, 'converted_teams']
control_n = conversion_counts.loc[control_idx, 'total_teams']
control_rate = pcr.loc[pcr['email_journey'] == 'Control', 'paid_conversion_rate'].values[0]

variant_a_conv = conversion_counts.loc[variant_a_idx, 'converted_teams']
variant_a_n = conversion_counts.loc[variant_a_idx, 'total_teams']
variant_a_rate = pcr.loc[pcr['email_journey'] == 'Variant_A', 'paid_conversion_rate'].values[0]

variant_b_conv = conversion_counts.loc[variant_b_idx, 'converted_teams']
variant_b_n = conversion_counts.loc[variant_b_idx, 'total_teams']
variant_b_rate = pcr.loc[pcr['email_journey'] == 'Variant_B', 'paid_conversion_rate'].values[0]

print(f"\n--- Pairwise Comparisons ---")

# Variant A vs Control
z_stat_a, p_value_a = proportions_ztest(
    count=[variant_a_conv, control_conv],
    nobs=[variant_a_n, control_n],
    alternative='two-sided'
)

print(f"\nVariant A vs Control (proportion z-test):")
print(f"  Control: {control_conv}/{control_n} = {control_rate:.1%}")
print(f"  Variant A: {variant_a_conv}/{variant_a_n} = {variant_a_rate:.1%}")
print(f"  Absolute difference: {(variant_a_rate - control_rate):.1%} points")
print(f"  Relative lift: {((variant_a_rate - control_rate) / control_rate * 100):+.1f}%")
print(f"  Z-statistic = {z_stat_a:.3f}")
print(f"  p-value = {p_value_a:.4f}")
print(f"  Significant: {'Yes ✓' if p_value_a < 0.05 else 'No'}")

# Variant B vs Control
z_stat_b, p_value_b = proportions_ztest(
    count=[variant_b_conv, control_conv],
    nobs=[variant_b_n, control_n],
    alternative='two-sided'
)

print(f"\nVariant B vs Control (proportion z-test):")
print(f"  Control: {control_conv}/{control_n} = {control_rate:.1%}")
print(f"  Variant B: {variant_b_conv}/{variant_b_n} = {variant_b_rate:.1%}")
print(f"  Absolute difference: {(variant_b_rate - control_rate):.1%} points")
print(f"  Relative lift: {((variant_b_rate - control_rate) / control_rate * 100):+.1f}%")
print(f"  Z-statistic = {z_stat_b:.3f}")
print(f"  p-value = {p_value_b:.4f}")
print(f"  Significant: {'Yes ✓' if p_value_b < 0.05 else 'No'}")


--- Pairwise Comparisons ---

Variant A vs Control (proportion z-test):
  Control: 4/40 = 10.0%
  Variant A: 10/40 = 25.0%
  Absolute difference: 15.0% points
  Relative lift: +150.0%
  Z-statistic = 1.765
  p-value = 0.0775
  Significant: No ✗

Variant B vs Control (proportion z-test):
  Control: 4/40 = 10.0%
  Variant B: 6/40 = 15.0%
  Absolute difference: 5.0% points
  Relative lift: +50.0%
  Z-statistic = 0.676
  p-value = 0.4990
  Significant: No ✗


# EFFECT SIZES (Cohen's h)

In [33]:
def cohens_h(p1, p2):
    """Calculate Cohen's h for two proportions"""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

h_a = cohens_h(variant_a_rate, control_rate)
h_b = cohens_h(variant_b_rate, control_rate)

print(f"\n--- Effect Sizes (Cohen's h) ---")
print(f"Variant A vs Control: h = {h_a:.3f} ({'small' if abs(h_a) < 0.5 else 'medium' if abs(h_a) < 0.8 else 'large'})")
print(f"Variant B vs Control: h = {h_b:.3f} ({'small' if abs(h_b) < 0.5 else 'medium' if abs(h_b) < 0.8 else 'large'})")


--- Effect Sizes (Cohen's h) ---
Variant A vs Control: h = 0.404 (small)
Variant B vs Control: h = 0.152 (small)


# CONFIDENCE INTERVALS

In [34]:
from statsmodels.stats.proportion import proportion_confint

# 95% confidence intervals
ci_control = proportion_confint(control_conv, control_n, alpha=0.05, method='wilson')
ci_variant_a = proportion_confint(variant_a_conv, variant_a_n, alpha=0.05, method='wilson')
ci_variant_b = proportion_confint(variant_b_conv, variant_b_n, alpha=0.05, method='wilson')

print(f"\n--- 95% Confidence Intervals ---")
print(f"Control:   {control_rate:.1%} [{ci_control[0]:.1%}, {ci_control[1]:.1%}]")
print(f"Variant A: {variant_a_rate:.1%} [{ci_variant_a[0]:.1%}, {ci_variant_a[1]:.1%}]")
print(f"Variant B: {variant_b_rate:.1%} [{ci_variant_b[0]:.1%}, {ci_variant_b[1]:.1%}]")


--- 95% Confidence Intervals ---
Control:   10.0% [4.0%, 23.1%]
Variant A: 25.0% [14.2%, 40.2%]
Variant B: 15.0% [7.1%, 29.1%]


# BUSINESS IMPACT CALCULATION

In [35]:
print(f"\n--- Business Impact Analysis ---")

# Assuming you continue with 1000 new teams per quarter
new_teams_per_quarter = 1000

# Additional conversions if you deploy each variant
additional_conv_a = new_teams_per_quarter * (variant_a_rate - control_rate)
additional_conv_b = new_teams_per_quarter * (variant_b_rate - control_rate)

print(f"\nIf deployed to 1,000 new teams per quarter:")
print(f"  Control baseline: {new_teams_per_quarter * control_rate:.0f} conversions")
print(f"  Variant A: {new_teams_per_quarter * variant_a_rate:.0f} conversions (+{additional_conv_a:.0f} incremental)")
print(f"  Variant B: {new_teams_per_quarter * variant_b_rate:.0f} conversions (+{additional_conv_b:.0f} incremental)")

# Assuming average ACV (you'll calculate this next)
assumed_acv = 2500  # placeholder
print(f"\nAssuming avg ACV of ${assumed_acv:,}:")
print(f"  Variant A incremental revenue: ${additional_conv_a * assumed_acv:,.0f} per quarter")
print(f"  Variant B incremental revenue: ${additional_conv_b * assumed_acv:,.0f} per quarter")

print("\n" + "="*80)


--- Business Impact Analysis ---

If deployed to 1,000 new teams per quarter:
  Control baseline: 100 conversions
  Variant A: 250 conversions (+150 incremental)
  Variant B: 150 conversions (+50 incremental)

Assuming avg ACV of $2,500:
  Variant A incremental revenue: $375,000 per quarter
  Variant B incremental revenue: $125,000 per quarter



# Average Contract Value (ACV)

In [37]:
acv = df_onboarding.merge(df_financial[['team_id','first_annual_contract_value_usd']], on='team_id', how='left')
acv_summary = acv.groupby('email_journey')['first_annual_contract_value_usd'].mean().reset_index()
print(acv_summary)

  email_journey  first_annual_contract_value_usd
0       Control                           1500.0
1     Variant_A                           2450.0
2     Variant_B                           4000.0


In [38]:
# Get only converted teams (non-null ACV values)
acv_converted = acv[acv['first_annual_contract_value_usd'].notna()].copy()

# Detailed statistics by journey
acv_stats = acv_converted.groupby('email_journey')['first_annual_contract_value_usd'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(2)

print("\nDetailed ACV Statistics (Converted Teams Only):")
print(acv_stats)

# Split data by journey for statistical tests
control_acv = acv_converted[acv_converted['email_journey'] == 'Control']['first_annual_contract_value_usd']
variant_a_acv = acv_converted[acv_converted['email_journey'] == 'Variant_A']['first_annual_contract_value_usd']
variant_b_acv = acv_converted[acv_converted['email_journey'] == 'Variant_B']['first_annual_contract_value_usd']

print(f"\nSample sizes (converted teams):")
print(f"  Control: n = {len(control_acv)}")
print(f"  Variant A: n = {len(variant_a_acv)}")
print(f"  Variant B: n = {len(variant_b_acv)}")


Detailed ACV Statistics (Converted Teams Only):
               count    mean  median    std     min     max
email_journey                                              
Control            4  1500.0  1500.0    0.0  1500.0  1500.0
Variant_A         10  2450.0  2500.0  437.8  2000.0  3000.0
Variant_B          6  4000.0  4000.0    0.0  4000.0  4000.0

Sample sizes (converted teams):
  Control: n = 4
  Variant A: n = 10
  Variant B: n = 6


# ONE-WAY ANOVA (Overall test for all 3 groups)

In [41]:
f_stat, p_value_anova = stats.f_oneway(control_acv, variant_a_acv, variant_b_acv)

print(f"\nOne-way ANOVA (3 groups):")
print(f"  F-statistic = {f_stat:.3f}")
print(f"  p-value = {p_value_anova:.4f}")
print(f"  Significant: {'Yes' if p_value_anova < 0.05 else 'No'}")



One-way ANOVA (3 groups):
  F-statistic = 81.366
  p-value = 0.0000
  Significant: Yes


# PAIRWISE T-TESTS (Independent samples)

In [40]:
# Variant A vs Control
t_stat_a, p_value_a = stats.ttest_ind(variant_a_acv, control_acv)

print(f"\nVariant A vs Control (independent t-test):")
print(f"  Control: ${control_acv.mean():,.0f} (n={len(control_acv)})")
print(f"  Variant A: ${variant_a_acv.mean():,.0f} (n={len(variant_a_acv)})")
print(f"  Absolute difference: ${(variant_a_acv.mean() - control_acv.mean()):+,.0f}")
print(f"  Relative lift: {((variant_a_acv.mean() - control_acv.mean()) / control_acv.mean() * 100):+.1f}%")
print(f"  t-statistic = {t_stat_a:.3f}")
print(f"  p-value = {p_value_a:.4f}")
print(f"  Significant: {'Yes ✓' if p_value_a < 0.05 else 'No'}")

# Variant B vs Control
t_stat_b, p_value_b = stats.ttest_ind(variant_b_acv, control_acv)

print(f"\nVariant B vs Control (independent t-test):")
print(f"  Control: ${control_acv.mean():,.0f} (n={len(control_acv)})")
print(f"  Variant B: ${variant_b_acv.mean():,.0f} (n={len(variant_b_acv)})")
print(f"  Absolute difference: ${(variant_b_acv.mean() - control_acv.mean()):+,.0f}")
print(f"  Relative lift: {((variant_b_acv.mean() - control_acv.mean()) / control_acv.mean() * 100):+.1f}%")
print(f"  t-statistic = {t_stat_b:.3f}")
print(f"  p-value = {p_value_b:.4f}")
print(f"  Significant: {'Yes ✓' if p_value_b < 0.05 else 'No'}")


Variant A vs Control (independent t-test):
  Control: $1,500 (n=4)
  Variant A: $2,450 (n=10)
  Absolute difference: $+950
  Relative lift: +63.3%
  t-statistic = 4.235
  p-value = 0.0012
  Significant: Yes ✓

Variant B vs Control (independent t-test):
  Control: $1,500 (n=4)
  Variant B: $4,000 (n=6)
  Absolute difference: $+2,500
  Relative lift: +166.7%
  t-statistic = inf
  p-value = 0.0000
  Significant: Yes ✓


# EFFECT SIZES (Cohen's d)

In [42]:
def cohens_d(group1, group2):
    """Calculate Cohen's d for two groups"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

d_a = cohens_d(variant_a_acv, control_acv)
d_b = cohens_d(variant_b_acv, control_acv)

print(f"\n--- Effect Sizes (Cohen's d) ---")
print(f"Variant A vs Control: d = {d_a:.3f} ({'small' if abs(d_a) < 0.5 else 'medium' if abs(d_a) < 0.8 else 'large'})")
print(f"Variant B vs Control: d = {d_b:.3f} ({'small' if abs(d_b) < 0.5 else 'medium' if abs(d_b) < 0.8 else 'large'})")


--- Effect Sizes (Cohen's d) ---
Variant A vs Control: d = 2.506 (large)
Variant B vs Control: d = inf (large)


C:\Users\John\AppData\Local\Temp\ipykernel_3700\507901846.py:6: RuntimeWarning: divide by zero encountered in scalar divide
  return (group1.mean() - group2.mean()) / pooled_std


# CONFIDENCE INTERVALS (95%)

In [43]:



from scipy.stats import t as t_dist

def calculate_ci(data, confidence=0.95):
    """Calculate confidence interval for mean"""
    n = len(data)
    mean = data.mean()
    se = data.std() / np.sqrt(n)
    margin = se * t_dist.ppf((1 + confidence) / 2, n - 1)
    return (mean - margin, mean + margin)

ci_control = calculate_ci(control_acv)
ci_variant_a = calculate_ci(variant_a_acv)
ci_variant_b = calculate_ci(variant_b_acv)

print(f"\n--- 95% Confidence Intervals ---")
print(f"Control:   ${control_acv.mean():,.0f} [${ci_control[0]:,.0f}, ${ci_control[1]:,.0f}]")
print(f"Variant A: ${variant_a_acv.mean():,.0f} [${ci_variant_a[0]:,.0f}, ${ci_variant_a[1]:,.0f}]")
print(f"Variant B: ${variant_b_acv.mean():,.0f} [${ci_variant_b[0]:,.0f}, ${ci_variant_b[1]:,.0f}]")


--- 95% Confidence Intervals ---
Control:   $1,500 [$1,500, $1,500]
Variant A: $2,450 [$2,137, $2,763]
Variant B: $4,000 [$4,000, $4,000]


# NON-PARAMETRIC TEST (Mann-Whitney U - if data is not normally distributed)

In [44]:
print(f"\n--- Non-Parametric Tests (Mann-Whitney U) ---")
print("(Use if ACV distribution is skewed)")

# Variant A vs Control
u_stat_a, p_value_u_a = stats.mannwhitneyu(variant_a_acv, control_acv, alternative='two-sided')
print(f"\nVariant A vs Control (Mann-Whitney U):")
print(f"  U-statistic = {u_stat_a:.3f}")
print(f"  p-value = {p_value_u_a:.4f}")

# Variant B vs Control
u_stat_b, p_value_u_b = stats.mannwhitneyu(variant_b_acv, control_acv, alternative='two-sided')
print(f"\nVariant B vs Control (Mann-Whitney U):")
print(f"  U-statistic = {u_stat_b:.3f}")
print(f"  p-value = {p_value_u_b:.4f}")


--- Non-Parametric Tests (Mann-Whitney U) ---
(Use if ACV distribution is skewed)

Variant A vs Control (Mann-Whitney U):
  U-statistic = 40.000
  p-value = 0.0044

Variant B vs Control (Mann-Whitney U):
  U-statistic = 24.000
  p-value = 0.0040


# BUSINESS IMPACT CALCULATION

In [46]:
# Revenue per 1000 teams (combining conversion rate + ACV)
control_pcr = 0.10
variant_a_pcr = 0.25
variant_b_pcr = 0.15

new_teams = 1000

control_revenue = new_teams * control_pcr * control_acv.mean()
variant_a_revenue = new_teams * variant_a_pcr * variant_a_acv.mean()
variant_b_revenue = new_teams * variant_b_pcr * variant_b_acv.mean()

print(f"\nExpected Revenue per 1,000 New Teams (Conversion × ACV):")
print(f"  Control:   {new_teams * control_pcr:.0f} conversions × ${control_acv.mean():,.0f} = ${control_revenue:,.0f}")
print(f"  Variant A: {new_teams * variant_a_pcr:.0f} conversions × ${variant_a_acv.mean():,.0f} = ${variant_a_revenue:,.0f} (+${variant_a_revenue - control_revenue:,.0f})")
print(f"  Variant B: {new_teams * variant_b_pcr:.0f} conversions × ${variant_b_acv.mean():,.0f} = ${variant_b_revenue:,.0f} (+${variant_b_revenue - control_revenue:,.0f})")




Expected Revenue per 1,000 New Teams (Conversion × ACV):
  Control:   100 conversions × $1,500 = $150,000
  Variant A: 250 conversions × $2,450 = $612,500 (+$462,500)
  Variant B: 150 conversions × $4,000 = $600,000 (+$450,000)


# DISTRIBUTION CHECK (Normality test - Shapiro-Wilk)

In [48]:
print("(If p < 0.05, data is NOT normally distributed → use Mann-Whitney instead of t-test)")

_, p_shapiro_control = stats.shapiro(control_acv)
_, p_shapiro_a = stats.shapiro(variant_a_acv)
_, p_shapiro_b = stats.shapiro(variant_b_acv)

print(f"  Control: p = {p_shapiro_control:.4f} {'(Not normal)' if p_shapiro_control < 0.05 else '(Normal)'}")
print(f"  Variant A: p = {p_shapiro_a:.4f} {'(Not normal)' if p_shapiro_a < 0.05 else '(Normal)'}")
print(f"  Variant B: p = {p_shapiro_b:.4f} {'(Not normal)' if p_shapiro_b < 0.05 else '(Normal)'}")

print("\n" + "="*80)

(If p < 0.05, data is NOT normally distributed → use Mann-Whitney instead of t-test)
  Control: p = 1.0000 (Normal)
  Variant A: p = 0.0167 (Not normal)
  Variant B: p = 1.0000 (Normal)

